# Medallion schema migration

One-time move of all objects from `capstone6_telecom.telecom` into `bronze`, `silver` and `gold` schemas, and of the external volume `ext_raw_landing` into `bronze`. Safe to re-run: every step checks what already exists.

In [0]:
C = "capstone6_telecom"
OLD = "telecom"
VOLUME = "ext_raw_landing"

def layer(name):
    if name.startswith("bronze_") or name in ("region_id", "tower_id", "profiling_summary"):
        return "bronze"
    if name.startswith("silver_") or name.startswith("test_cdc_"):
        return "silver"
    return "gold"   # dim_*, fact_*, mart_*, gold_reconciliation

for s in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {C}.{s} COMMENT '{s} layer (medallion)'")

In [0]:
# ALTER ... RENAME keeps data, Delta history and table id (streaming checkpoints stay valid)
objs = spark.sql(f"SELECT table_name, table_type FROM {C}.information_schema.tables WHERE table_schema = '{OLD}'").collect()
for r in objs:
    if r.table_type == "VIEW":
        continue   # views are recreated below (a view cannot be renamed once its source table moved)
    spark.sql(f"ALTER TABLE {C}.{OLD}.{r.table_name} RENAME TO {C}.{layer(r.table_name)}.{r.table_name}")
    print(f"TABLE {r.table_name} -> {layer(r.table_name)}")

# The view stores its SQL text, so recreate it in gold pointing at the gold table
spark.sql(f"""CREATE OR REPLACE VIEW {C}.gold.mart_network_daily_overall AS
SELECT event_date, SUM(readings) AS readings, SUM(outage_readings) AS outage_readings,
       ROUND(100.0 * (SUM(readings) - SUM(outage_readings)) / SUM(readings), 2) AS availability_pct,
       ROUND(SUM(packet_loss_sum) / SUM(readings), 3) AS avg_packet_loss_pct,
       SUM(outage_minutes) AS outage_minutes
FROM {C}.gold.mart_network_daily
GROUP BY event_date""")

In [0]:
# A volume cannot be renamed into another schema. Dropping an EXTERNAL volume only removes the
# Unity Catalog registration; files (data, _schemas, _checkpoints) stay in ADLS and are re-registered in bronze.
old = spark.sql(f"SELECT storage_location, volume_type FROM {C}.information_schema.volumes "
                f"WHERE volume_schema = '{OLD}' AND volume_name = '{VOLUME}'").collect()
if old:
    loc, vtype = old[0]
    assert vtype == "EXTERNAL", "Only external volumes are moved this way"
    print("Files before:", [f.name for f in dbutils.fs.ls(f"/Volumes/{C}/{OLD}/{VOLUME}/")])
    spark.sql(f"DROP VOLUME {C}.{OLD}.{VOLUME}")
    spark.sql(f"CREATE EXTERNAL VOLUME {C}.bronze.{VOLUME} LOCATION '{loc}'")
print("Files now:", [f.name for f in dbutils.fs.ls(f"/Volumes/{C}/bronze/{VOLUME}/")])

In [0]:
display(spark.sql(f"""
SELECT 'table' AS kind, table_schema AS schema, table_name AS name, table_type AS type
FROM {C}.information_schema.tables WHERE table_schema IN ('{OLD}', 'bronze', 'silver', 'gold')
UNION ALL
SELECT 'volume', volume_schema, volume_name, volume_type FROM {C}.information_schema.volumes
ORDER BY schema, kind, name"""))

In [0]:
# The old telecom copy of the view points at a table that moved to gold, so it can no longer be queried.
# The working version is capstone6_telecom.gold.mart_network_daily_overall.
# The telecom schema itself is kept until teammates' copies of the repo stop writing to it.
spark.sql(f"DROP VIEW IF EXISTS {C}.{OLD}.mart_network_daily_overall")
display(spark.sql(f"SELECT table_name FROM {C}.information_schema.tables WHERE table_schema = '{OLD}'"))